# Real-Load Test — Lausanne Region (1 000 pairs)

Simulates realistic API traffic against the Lausanne-region planner.

**Pair distribution** mirrors how real users query a transit planner:
- **50 %** hub ↔ hub (top-30 busy stops — high contention, repeated pairs expected)
- **25 %** hub ↔ secondary (top-30 → top 31–150)
- **15 %** secondary ↔ secondary
- **10 %** fully random from any stop

Because popular hub pairs are overrepresented, the TTL cache absorbs many
duplicate calls — exactly as it would in production.

**Workflow**
1. Run **Setup** once per kernel session.
2. Run **Prepare planner** (only needed once; skip if already prepared).
3. Run **Generate pairs** → **Benchmark** → **Results**.

## Setup

In [4]:
import importlib
import os
import sys

cwd = os.path.abspath(os.getcwd())
project_root = cwd if os.path.exists(os.path.join(cwd, "src")) else os.path.abspath(os.path.join(cwd, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import src.config.settings as _s
import src.data.csa_data_handler as _cdh
import src.models.delay_model as _dm
import src.models.delay_model_trainer as _dmt
import src.models.model_artifacts as _ma
import src.routing.robust_journey_planner as _rjp
import tests.test_robust_journey as _bench


def _prepared_summary(p):
    if p is None or not getattr(p, "prepared", False):
        return " (robust planner is not prepared yet)"
    days = getattr(p, "connections_by_day", None) or {}
    n_day_connections = sum(len(v) for v in days.values())
    n_stops = len(getattr(p, "stops", None) or [])
    n_trips = getattr(p, "n_trips", 0)
    has_delays = bool(getattr(p, "delay_lookup", None))
    return f" ({n_stops:,} stops, {n_trips:,} trips, {n_day_connections:,} day-connections, delay_lookup={has_delays})"


def reload_robust_code(preserve_prepared=True):
    global settings, get_settings, RobustJourneyPlanner
    global bench_robust_prepare, bench_robust_plan
    global _s, _cdh, _dm, _dmt, _ma, _rjp, _bench, robust_planner

    old_planner = globals().get("robust_planner")
    for module in (_s, _cdh, _dm, _dmt, _ma, _rjp, _bench):
        importlib.reload(module)

    get_settings = _s.get_settings
    RobustJourneyPlanner = _rjp.RobustJourneyPlanner
    bench_robust_prepare = _bench.bench_robust_prepare
    bench_robust_plan = _bench.bench_robust_plan

    try:
        settings = get_settings()
    except Exception:
        if old_planner is not None and hasattr(old_planner, "settings"):
            settings = old_planner.settings
        elif "settings" in globals():
            settings = globals()["settings"]
        else:
            raise

    new_planner = RobustJourneyPlanner(settings=settings)
    if preserve_prepared and old_planner is not None and getattr(old_planner, "prepared", False):
        for name, value in vars(old_planner).items():
            setattr(new_planner, name, value)
        new_planner.settings = settings
        if getattr(new_planner, "data_handler", None) is not None:
            try:
                new_planner.data_handler.__class__ = _cdh.CSADataHandler
            except TypeError:
                pass
        robust_planner = new_planner
        print("Reloaded robust_journey_planner.py; preserved prepared data" + _prepared_summary(robust_planner))
    else:
        robust_planner = new_planner
        print("Reloaded robust_journey_planner.py; created a fresh unprepared robust planner")
    return robust_planner


robust_planner = reload_robust_code(preserve_prepared=True)

Reloaded robust_journey_planner.py; created a fresh unprepared robust planner


In [11]:
# Reload latest code while keeping prepared data in memory.
robust_planner = reload_robust_code()

Reloaded robust_journey_planner.py; preserved prepared data (25,752 stops, 1,340,906 trips, 35,752,786 day-connections, delay_lookup=True)


In [18]:
LAUSANNE_REGION_UUIDS = (
    "a7a21b73-6ffe-4fbf-a635-6e2b961f3072",
    "e168fd57-f57a-4075-a350-0dcfbb55147f",
)
REGION_UUIDS          = settings.region_uuids or LAUSANNE_REGION_UUIDS
REGION_LABEL          = "Lausanne"

TRAVEL_DATE           = "2026-05-27"
DEADLINE              = "18:00"
# Confidence sampled per query — range 0.5–0.9, skewed towards the lower end
CONFIDENCE_LEVELS  = [0.50, 0.60, 0.70, 0.75, 0.80, 0.90]
CONFIDENCE_WEIGHTS = [30,   25,   20,   15,    7,    3   ]
MAX_ROUTES            = 3
SEARCH_WINDOW_MINUTES = 120
N_PAIRS               = 1000

print(f"Region        : {REGION_LABEL}")
print(f"Regions       : {REGION_UUIDS}")
print(f"Travel date   : {TRAVEL_DATE}")
print(f"Deadline      : {DEADLINE}")
print(f"Confidence    : {CONFIDENCE_LEVELS}")
print(f"  weights     : {CONFIDENCE_WEIGHTS}")
print(f"Max routes    : {MAX_ROUTES}")
print(f"Search window : {SEARCH_WINDOW_MINUTES} min")
print(f"Pairs         : {N_PAIRS}")

Region        : Lausanne
Regions       : ('a7a21b73-6ffe-4fbf-a635-6e2b961f3072', 'e168fd57-f57a-4075-a350-0dcfbb55147f')
Travel date   : 2026-05-27
Deadline      : 18:00
Confidence    : [0.5, 0.6, 0.7, 0.75, 0.8, 0.9]
  weights     : [30, 25, 20, 15, 7, 3]
Max routes    : 3
Search window : 120 min
Pairs         : 1000


## Prepare planner

Skip this cell if the planner is already prepared (`robust_planner.prepared == True`).
Set `FORCE_PREPARE = True` to rebuild Trino tables and reload all data.

In [13]:
FORCE_PREPARE             = True
FORCE_REBUILD             = True
REBUILD_CSA_PREREQUISITES = True

if robust_planner.prepared and not FORCE_PREPARE:
    print("robust_planner already prepared; skipping. Set FORCE_PREPARE=True to run again.")
else:
    robust_planner = reload_robust_code(preserve_prepared=False)
    bench_robust_prepare(
        robust_planner,
        regions=REGION_UUIDS,
        force_rebuild=FORCE_REBUILD,
        rebuild_csa_prerequisites=REBUILD_CSA_PREREQUISITES,
        load_delay_lookup=True,
        travel_date=TRAVEL_DATE,
    )

Reloaded robust_journey_planner.py; created a fresh unprepared robust planner

  bench_robust_prepare
  LOAD DELAY LOOKUP
  Loaded 2/2 district files
  delay_lookup.load              24.18s  (803706 keys)

  delay model loaded  (7 quantile boosters)

  BUILD CSA TABLES
  creating  stops...
  build_stops                     0.95s


/home/kuci/project/final/src/data/footpaths_data.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


  creating  stop_to_stop...
  build_footpaths                 1.25s
  creating  stop_times_seq...
  build_stop_times_seq            3.60s
  creating  stop_times_trips_seq...
  build_stop_times_trips_seq      5.19s
  creating  full_table_seq...
  build_full_table_seq            4.08s
  creating  connections...
  build_connections               4.51s
--------------------------------------------
  total build_all                19.59s

  FETCH DATA
  fetch_stops                     0.07s  (390 rows)
  fetch_footpaths                 0.09s  (1492 rows)
  fetch_connections              18.09s  (620440 rows)

  MATERIALIZE IN-MEMORY
  stops dict                      0.00s  (390 stops)
  footpaths dict                  0.00s  (1492 edges)
  connections loop                3.29s  (620440 rows)
  sort + columns                  3.16s  (52529 trips)
  trip meta by idx                0.02s
--------------------------------------------
  TOTAL prepare()                75.61s

  BAKE QUANTILES — 202

In [14]:
planner = robust_planner
print(f"prepared     = {planner.prepared}")
print(f"delay_lookup = {bool(planner.delay_lookup)}")
print(f"n_stops      = {len(planner.stops)}")
print(f"n_trips      = {planner.n_trips}")

prepared     = True
delay_lookup = True
n_stops      = 390
n_trips      = 52529


## Generate realistic test pairs

Stops are ranked by how often they appear across weekday connections (Mon–Fri).

| Tier | Definition | Role |
|------|-----------|------|
| Hub (top 30) | major interchange stops | 50 % hub↔hub + half of the 25 % hub↔mid |
| Mid (31–150) | secondary stops | other half of hub↔mid + 15 % mid↔mid |
| Low (151+)   | local stops | part of 10 % random |

Duplicated pairs arise naturally from the skewed distribution, matching
the real workload pattern where the cache absorbs repeated popular-route requests.

In [15]:
import random
import statistics
from collections import Counter

stop_freq = Counter()
for day in ["monday", "tuesday", "wednesday", "thursday", "friday"]:
    for conn in planner.connections_by_day[day]:
        stop_freq[conn[0]] += 1
        stop_freq[conn[1]] += 1

ranked   = [s for s, _ in stop_freq.most_common()]
n_all    = len(ranked)
top_tier = ranked[:30]
mid_tier = ranked[30:min(150, n_all)]

print(f"Ranked stops : {n_all} total")
print(f"  hub  (top 30)   : {len(top_tier)}")
print(f"  mid  (31-150)   : {len(mid_tier)}")
print(f"  low  (151+)     : {n_all - len(top_tier) - len(mid_tier)}")

rng         = random.Random(42)
TEST_PAIRS  = []
TEST_CONFS  = []
tier_labels = []

while len(TEST_PAIRS) < N_PAIRS:
    r = rng.random()
    if r < 0.50:
        s, e  = rng.choice(top_tier), rng.choice(top_tier)
        label = "hub↔hub"
    elif r < 0.75:
        s = rng.choice(top_tier)
        e = rng.choice(mid_tier) if mid_tier else rng.choice(ranked)
        if rng.random() < 0.5:
            s, e = e, s
        label = "hub↔mid"
    elif r < 0.90:
        s = rng.choice(mid_tier) if mid_tier else rng.choice(ranked)
        e = rng.choice(mid_tier) if mid_tier else rng.choice(ranked)
        label = "mid↔mid"
    else:
        s, e  = rng.choice(ranked), rng.choice(ranked)
        label = "random"
    if s != e:
        TEST_PAIRS.append((s, e))
        TEST_CONFS.append(rng.choices(CONFIDENCE_LEVELS, weights=CONFIDENCE_WEIGHTS, k=1)[0])
        tier_labels.append(label)

n_unique = len(set(TEST_PAIRS))
print(f"\nGenerated {len(TEST_PAIRS)} pairs:")
for cat, cnt in Counter(tier_labels).most_common():
    print(f"  {cat:<20} {cnt:>4}  ({100 * cnt / N_PAIRS:.0f}%)")
print(f"\nUnique pairs : {n_unique}")
print(f"Duplicates   : {len(TEST_PAIRS) - n_unique}")

conf_dist = Counter(TEST_CONFS)
print(f"\nConfidence distribution:")
for lv in CONFIDENCE_LEVELS:
    cnt = conf_dist.get(lv, 0)
    bar = "█" * (cnt // 10)
    print(f"  q={lv:.2f}  {cnt:>4}  ({100 * cnt / N_PAIRS:.0f}%)  {bar}")

Ranked stops : 389 total
  hub  (top 30)   : 30
  mid  (31-150)   : 120
  low  (151+)     : 239

Generated 1000 pairs:
  hub↔hub               486  (49%)
  hub↔mid               258  (26%)
  mid↔mid               144  (14%)
  random                112  (11%)

Unique pairs : 888
Duplicates   : 112

Confidence distribution:
  q=0.50   316  (32%)  ███████████████████████████████
  q=0.60   237  (24%)  ███████████████████████
  q=0.70   191  (19%)  ███████████████████
  q=0.75   158  (16%)  ███████████████
  q=0.80    72  (7%)  ███████
  q=0.90    26  (3%)  ██


## Benchmark — 1 000 queries

In [20]:
import time

robust_planner._route_cache.clear()   # start cold; no pre-warmed cache entries

all_times  = []
all_routes = []
no_route   = 0

for idx, ((start_stop, end_stop), conf_q) in enumerate(zip(TEST_PAIRS, TEST_CONFS)):
    t0 = time.perf_counter()
    result = robust_planner.plan(
        start_stop_id=start_stop,
        end_stop_id=end_stop,
        travel_date=TRAVEL_DATE,
        arrival_deadline=DEADLINE,
        confidence_q=conf_q,
        max_routes=MAX_ROUTES,
        search_window_minutes=SEARCH_WINDOW_MINUTES,
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000
    all_times.append(elapsed_ms)
    n = len(result)
    all_routes.append(n)
    if n == 0:
        no_route += 1
    if (idx + 1) % 100 == 0:
        print(f"  [{idx + 1:>4}/{N_PAIRS}]  running avg: {statistics.mean(all_times):.1f} ms")

print(f"\nFinished {len(all_times)} queries.")

  plan                total=53.2ms  scanned=13951  routes=3
  plan                total=10.9ms  scanned=9812  routes=3
  plan                total=3.0ms  scanned=3339  routes=3
  plan                total=105.7ms  scanned=24220  routes=3
  plan                total=32.3ms  scanned=18326  routes=3
  plan                total=137.1ms  scanned=34071  routes=3
  plan                total=20.6ms  scanned=11622  routes=3
  plan                total=6.1ms  scanned=5503  routes=3
  plan                total=108.7ms  scanned=33221  routes=3
  plan                total=23.3ms  scanned=13172  routes=3
  plan                total=8.6ms  scanned=8128  routes=3
  plan                total=5.3ms  scanned=5735  routes=3
  plan                total=9.4ms  scanned=8691  routes=3
  plan                total=39.9ms  scanned=18238  routes=3
  plan                total=51.3ms  scanned=21514  routes=3
  plan                total=12.3ms  scanned=8675  routes=3
  plan                total=93.0ms  scanned=34560

## Results

In [21]:
import pandas as pd

def percentile(lst, p):
    k = max(0, min(int(len(lst) * p / 100), len(lst) - 1))
    return lst[k]

sorted_times = sorted(all_times)
n            = len(sorted_times)
cache_hits   = sum(1 for t in all_times if t < 1.0)
conf_dist    = Counter(TEST_CONFS)

settings_data = [
    ("Region",           REGION_LABEL),
    ("Travel date",      TRAVEL_DATE),
    ("Deadline",         DEADLINE),
    ("Confidence range", "0.50–0.90 (skewed low)"),
    ("Max routes",       str(MAX_ROUTES)),
    ("Search window",    f"{SEARCH_WINDOW_MINUTES} min"),
    ("Total pairs",      str(N_PAIRS)),
    ("Unique pairs",     str(len(set(TEST_PAIRS)))),
    ("Duplicates",       str(len(TEST_PAIRS) - len(set(TEST_PAIRS)))),
]

timing_data = [
    ("avg",    f"{statistics.mean(all_times):.2f} ms"),
    ("median", f"{percentile(sorted_times, 50):.2f} ms"),
    ("p75",    f"{percentile(sorted_times, 75):.2f} ms"),
    ("p95",    f"{percentile(sorted_times, 95):.2f} ms"),
    ("p99",    f"{percentile(sorted_times, 99):.2f} ms"),
    ("max",    f"{sorted_times[-1]:.2f} ms"),
    ("stdev",  f"{statistics.stdev(all_times):.2f} ms"),
]

quality_data = [
    ("avg routes",  f"{statistics.mean(all_routes):.2f}"),
    ("no-route %",  f"{100 * no_route / n:.1f}%"),
    ("cache hits",  f"{cache_hits}  ({100 * cache_hits / n:.0f}% of calls < 1 ms)"),
]

dist_data = [(cat, cnt, f"{100 * cnt / N_PAIRS:.0f}%")
             for cat, cnt in Counter(tier_labels).most_common()]

conf_data = [(f"q={lv:.2f}", conf_dist.get(lv, 0), f"{100 * conf_dist.get(lv, 0) / N_PAIRS:.0f}%")
             for lv in CONFIDENCE_LEVELS]

SEP = "=" * 44

print(SEP)
print("  TEST SETTINGS")
print(SEP)
for k, v in settings_data:
    print(f"  {k:<22} {v}")

print()
print(SEP)
print("  TIMING RESULTS")
print(SEP)
for k, v in timing_data:
    print(f"  {k:<22} {v}")

print()
print(SEP)
print("  ROUTE QUALITY")
print(SEP)
for k, v in quality_data:
    print(f"  {k:<22} {v}")

print()
print(SEP)
print("  PAIR DISTRIBUTION")
print(SEP)
for cat, cnt, pct in dist_data:
    print(f"  {cat:<22} {cnt:>4}  ({pct})")

print()
print(SEP)
print("  CONFIDENCE DISTRIBUTION")
print(SEP)
for label, cnt, pct in conf_data:
    bar = "█" * (cnt // 10)
    print(f"  {label}  {cnt:>4}  ({pct})  {bar}")

print()
df_settings = pd.DataFrame(settings_data, columns=["Setting",    "Value"]).set_index("Setting")
df_timing   = pd.DataFrame(timing_data,   columns=["Metric",     "Value"]).set_index("Metric")
df_quality  = pd.DataFrame(quality_data,  columns=["Metric",     "Value"]).set_index("Metric")
df_dist     = pd.DataFrame(dist_data,     columns=["Category",   "Count", "Share"]).set_index("Category")
df_conf     = pd.DataFrame(conf_data,     columns=["Confidence", "Count", "Share"]).set_index("Confidence")

display(df_settings)
display(df_timing)
display(df_quality)
display(df_dist)
display(df_conf)

  TEST SETTINGS
  Region                 Lausanne
  Travel date            2026-05-27
  Deadline               18:00
  Confidence range       0.50–0.90 (skewed low)
  Max routes             3
  Search window          120 min
  Total pairs            1000
  Unique pairs           888
  Duplicates             112

  TIMING RESULTS
  avg                    57.80 ms
  median                 49.24 ms
  p75                    90.97 ms
  p95                    136.50 ms
  p99                    160.48 ms
  max                    179.82 ms
  stdev                  43.40 ms

  ROUTE QUALITY
  avg routes             2.94
  no-route %             1.0%
  cache hits             23  (2% of calls < 1 ms)

  PAIR DISTRIBUTION
  hub↔hub                 486  (49%)
  hub↔mid                 258  (26%)
  mid↔mid                 144  (14%)
  random                  112  (11%)

  CONFIDENCE DISTRIBUTION
  q=0.50   316  (32%)  ███████████████████████████████
  q=0.60   237  (24%)  ███████████████████████
  q

,Value
Setting,
Region,Lausanne
Travel date,2026-05-27
Deadline,18:00
Confidence range,0.50–0.90 (skewed low)
Max routes,3
Search window,120 min
Total pairs,1000
Unique pairs,888
Duplicates,112


,Value
Metric,
avg,57.80 ms
median,49.24 ms
p75,90.97 ms
p95,136.50 ms
p99,160.48 ms
max,179.82 ms
stdev,43.40 ms


,Value
Metric,
avg routes,2.94
no-route %,1.0%
cache hits,23 (2% of calls < 1 ms)


,Count,Share
Category,,
hub↔hub,486,49%
hub↔mid,258,26%
mid↔mid,144,14%
random,112,11%


,Count,Share
Confidence,,
q=0.50,316,32%
q=0.60,237,24%
q=0.70,191,19%
q=0.75,158,16%
q=0.80,72,7%
q=0.90,26,3%
